# A/B 테스트 검증

Qwen2.5 + Qwen3.5 결과 통합 평가 (Rule Pass Rate + Style Score)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/project_05

In [ ]:
# ============================================================
# A/B TEST AUTO EVALUATION
# Qwen2.5 + Qwen3.5 결과 통합 평가
# ============================================================

!pip install -q sentence-transformers scikit-learn

import re
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# 결과 파일
# ============================================================

CSV_FILES = [
    "/content/drive/MyDrive/project_05/csv/eval_outputs_ab_v2_final.csv",         # Qwen2.5
    "/content/drive/MyDrive/project_05/csv/eval_outputs_qwen35_ab_final.csv",     # Qwen3.5
]

# ============================================================
# 로드
# ============================================================

dfs = []

for path in CSV_FILES:
    temp = pd.read_csv(path)
    dfs.append(temp)

df = pd.concat(
    dfs,
    ignore_index=True,
)

print("total rows:", len(df))

# ============================================================
# Similarity Model
# ============================================================

print("loading similarity model...")

sim_model = SentenceTransformer(
    "jhgan/ko-sroberta-multitask"
)

print("similarity model loaded")

# ============================================================
# 평가 함수
# ============================================================

def korean_ratio(text):

    text = str(text)

    if len(text) == 0:
        return 0

    korean = len(
        re.findall(
            r"[가-힣]",
            text,
        )
    )

    return korean / len(text)


def chinese_detected(text):

    text = str(text)

    return bool(
        re.search(
            r"[\u4e00-\u9fff]",
            text,
        )
    )


def count_questions(text):

    text = str(text)

    count = text.count("?")
    count += text.count("？")

    return max(1, count)


def prompt_leakage(text):

    text = str(text).lower()

    leakage_keywords = [
        "global rule",
        "persona rule",
        "<think>",
        "</think>",
        "follow-up instruction"
    ]

    return any(
        keyword in text
        for keyword in leakage_keywords
    )


def pressure_style_pass(text):

    text = str(text)

    keywords = [
        "왜",
        "어떻게",
        "근거",
        "수치",
        "사례",
        "구체적",
        "경험",
    ]

    return any(
        keyword in text
        for keyword in keywords
    )


def friendly_style_pass(text):

    text = str(text)

    keywords = [
        "설명해 주실",
        "공유해 주실",
        "말씀해 주실",
        "경험",
        "역할",
        "과정",
        "배운",
    ]

    return any(
        keyword in text
        for keyword in keywords
    )


def single_question_pass(text):

    return count_questions(text) == 1


def hard_fail(row):

    if row["chinese_detected"]:
        return True

    if row["question_count"] > 1:
        return True

    if row["prompt_leakage"] and row["question_count"] > 2:
        return True  # 조건 완화

    return False

# ============================================================
# Reference Similarity
# ============================================================

def calc_similarity(ref_text, pred_text):

    ref_text = str(ref_text)
    pred_text = str(pred_text)

    if len(ref_text.strip()) == 0:
        return 0.0

    embeddings = sim_model.encode(
        [ref_text, pred_text],
        convert_to_numpy=True,
    )

    score = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]],
    )[0][0]

    return float(score)

# ============================================================
# 채점
# ============================================================

print("evaluating...")

df["korean_ratio"] = (
    df["model_output"]
    .apply(korean_ratio)
)

df["chinese_detected"] = (
    df["model_output"]
    .apply(chinese_detected)
)

df["question_count"] = (
    df["model_output"]
    .apply(count_questions)
)

df["single_question_pass"] = (
    df["model_output"]
    .apply(single_question_pass)
)

df["prompt_leakage"] = (
    df["model_output"]
    .apply(prompt_leakage)
)

df["pressure_style_pass"] = (
    df["model_output"]
    .apply(pressure_style_pass)
)

df["friendly_style_pass"] = (
    df["model_output"]
    .apply(friendly_style_pass)
)

df["hard_fail"] = (
    df.apply(
        hard_fail,
        axis=1,
    )
)

# ============================================================
# Persona Style Score
# ============================================================

df["style_pass"] = False

df.loc[
    df["persona"] == "pressure_interviewer",
    "style_pass"
] = df.loc[
    df["persona"] == "pressure_interviewer",
    "pressure_style_pass"
]

df.loc[
    df["persona"] == "friendly_interviewer",
    "style_pass"
] = df.loc[
    df["persona"] == "friendly_interviewer",
    "friendly_style_pass"
]

# ============================================================
# Similarity 계산
# ============================================================

print("calculating reference similarity...")

df["reference_similarity"] = df.apply(
    lambda row: calc_similarity(
        row["reference_output"],
        row["model_output"],
    ),
    axis=1,
)

print("similarity done")

# ============================================================
# Summary
# ============================================================

summary = (
    df.groupby(
        [
            "model_type",
            "model_name",
            "prompt_type",
            "persona",
        ]
    )
    .agg(
        total=("model_output", "count"),

        avg_reference_similarity=(
            "reference_similarity",
            "mean",
        ),

        avg_korean_ratio=(
            "korean_ratio",
            "mean",
        ),

        chinese_rate=(
            "chinese_detected",
            "mean",
        ),

        avg_question_count=(
            "question_count",
            "mean",
        ),

        single_question_rate=(
            "single_question_pass",
            "mean",
        ),

        prompt_leakage_rate=(
            "prompt_leakage",
            "mean",
        ),

        style_pass_rate=(
            "style_pass",
            "mean",
        ),

        hard_fail_rate=(
            "hard_fail",
            "mean",
        ),
    )
    .reset_index()
)

# ============================================================
# Percent 변환
# ============================================================

for col in [
    "avg_reference_similarity",
    "avg_korean_ratio",
    "chinese_rate",
    "single_question_rate",
    "prompt_leakage_rate",
    "style_pass_rate",
    "hard_fail_rate",
]:
    summary[col] = (
        summary[col] * 100
    ).round(2)

summary["avg_question_count"] = (
    summary["avg_question_count"]
    .round(2)
)

# ============================================================
# 저장
# ============================================================

SUMMARY_PATH = (
    "/content/drive/MyDrive/project_05/eval_summary_all_models.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print()
print("===== SUMMARY =====")
print(summary)

# ============================================================
# Hard Fail Samples
# ============================================================

fail_df = df[
    df["hard_fail"] == True
]

FAIL_PATH = (
    "/content/drive/MyDrive/project_05/hard_fail_samples_all_models.csv"
)

fail_df.to_csv(
    FAIL_PATH,
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# Best Cases
# ============================================================

best_cases = df[
    (df["hard_fail"] == False)
    &
    (df["single_question_pass"] == True)
    &
    (df["style_pass"] == True)
]

best_cases = best_cases.sample(
    min(50, len(best_cases)),
    random_state=42,
)

BEST_PATH = (
    "/content/drive/MyDrive/project_05/best_cases_all_models.csv"
)

best_cases.to_csv(
    BEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# Worst Cases
# ============================================================

worst_cases = df[
    df["hard_fail"] == True
]

WORST_PATH = (
    "/content/drive/MyDrive/project_05/worst_cases_all_models.csv"
)

worst_cases.to_csv(
    WORST_PATH,
    index=False,
    encoding="utf-8-sig",
)

# ============================================================
# 출력
# ============================================================

print()
print("saved files")
print("- summary :", SUMMARY_PATH)
print("- hard fail :", FAIL_PATH)
print("- best cases :", BEST_PATH)
print("- worst cases :", WORST_PATH)

print()
print("hard fail count :", len(fail_df))
print("best case count :", len(best_cases))
print("worst case count :", len(worst_cases))

In [ ]:
# ============================================================
# PROMPT / RESPONSE LENGTH ANALYSIS (ADDITIONAL CELL)
# ============================================================

from transformers import AutoTokenizer

print("loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct"
)

print("tokenizer loaded")


# ============================================================
# length 계산
# ============================================================

def extract_prompt_text(x):
    if isinstance(x, str):
        try:
            x = json.loads(x)
        except:
            return x
    return x.get("question", "") + " " + x.get("candidate_answer", "")

def calc_token_length(text):
    if text is None:
        return 0
    return len(tokenizer.encode(str(text)))


df["prompt_text"] = df["question"].fillna("") + " " + df["candidate_answer"].fillna("")

df["prompt_length"] = df["prompt_text"].apply(calc_token_length)

df["response_length"] = df["model_output"].apply(calc_token_length)


# ============================================================
# summary (groupby)
# ============================================================

length_summary = (
    df.groupby(
        ["model_type", "model_name", "prompt_type", "persona"]
    )
    .agg(
        total=("model_output", "count"),

        avg_prompt_length=("prompt_length", "mean"),
        avg_response_length=("response_length", "mean"),

        std_prompt_length=("prompt_length", "std"),
        std_response_length=("response_length", "std"),
    )
    .reset_index()
)


# ============================================================
# percent formatting 없음 (길이는 raw 유지)
# ============================================================

length_summary["avg_prompt_length"] = length_summary["avg_prompt_length"].round(2)
length_summary["avg_response_length"] = length_summary["avg_response_length"].round(2)


# ============================================================
# 출력
# ============================================================

print("\n===== LENGTH ANALYSIS SUMMARY =====")
display(length_summary)


# ============================================================
# 저장
# ============================================================

SAVE_PATH = "/content/drive/MyDrive/project_05/length_analysis_summary.csv"

length_summary.to_csv(
    SAVE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nsaved to:", SAVE_PATH)

In [ ]:
# ============================================================
# RESPONSE TIME ANALYSIS (ROBUST VERSION)
# ============================================================

import numpy as np

print("===== RESPONSE TIME ANALYSIS =====")

# ============================================================
# 0. response_time 존재 여부 체크
# ============================================================

has_latency = "response_time" in df.columns

if not has_latency:
    print("[WARN] response_time not found → switching to proxy analysis")

    # proxy latency: response length 기반
    df["response_time_proxy"] = df["model_output"].apply(
        lambda x: len(str(x)) / 20  # rough TPS assumption
    )

    time_col = "response_time_proxy"
else:
    time_col = "response_time"

# ============================================================
# 1. 기본 통계
# ============================================================

latency_summary = (
    df.groupby(["model_type", "model_name", "prompt_type", "persona"])
    .agg(
        total=("model_output", "count"),
        avg_response_time=(time_col, "mean"),
        std_response_time=(time_col, "std"),
        min_response_time=(time_col, "min"),
        max_response_time=(time_col, "max"),
    )
    .reset_index()
)

# rounding
for col in [
    "avg_response_time",
    "std_response_time",
    "min_response_time",
    "max_response_time",
]:
    latency_summary[col] = latency_summary[col].round(3)

# ============================================================
# 2. TPS (가능하면만)
# ============================================================

if has_latency and "response_tokens" in df.columns:

    df["tokens_per_sec"] = df["response_tokens"] / df["response_time"]

    tps_summary = (
        df.groupby(["model_type", "model_name", "prompt_type", "persona"])
        .agg(
            avg_tps=("tokens_per_sec", "mean"),
            std_tps=("tokens_per_sec", "std"),
        )
        .reset_index()
    )

    latency_summary = latency_summary.merge(
        tps_summary,
        on=["model_type", "model_name", "prompt_type", "persona"],
        how="left"
    )

else:
    latency_summary["avg_tps"] = np.nan
    latency_summary["std_tps"] = np.nan

# ============================================================
# 3. 출력
# ============================================================

print("\n===== LATENCY SUMMARY =====")
display(latency_summary)

# ============================================================
# 4. 저장
# ============================================================

SAVE_PATH = "/content/drive/MyDrive/project_05/latency_analysis_summary.csv"

latency_summary.to_csv(
    SAVE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved to:", SAVE_PATH)

In [ ]:
# ============================================================
# SOFT FAILURE + CONTINUOUS METRICS EXTENSION
# ============================================================

import numpy as np

print("building soft metrics...")

# ============================================================
# 1. LEAKAGE SEVERITY SCORE (0~3)
# ============================================================

def leakage_severity(text):
    text = str(text).lower()

    if "<think>" in text or "</think>" in text:
        return 3
    if "rule" in text or "instruction" in text:
        return 2
    if "follow-up" in text or "꼬리질문" in text:
        return 1
    return 0


df["leakage_severity"] = df["model_output"].apply(leakage_severity)

# ============================================================
# 2. STYLE SCORE (continuous instead of binary)
# ============================================================

pressure_keywords = ["왜", "어떻게", "근거", "수치", "사례", "경험", "구체적"]
friendly_keywords = ["설명해", "공유해", "말씀해", "경험", "과정", "배운", "역할"]

def style_score(text, keywords):
    text = str(text)
    return sum(k in text for k in keywords) / len(keywords)


df["pressure_style_score"] = df["model_output"].apply(
    lambda x: style_score(x, pressure_keywords)
)

df["friendly_style_score"] = df["model_output"].apply(
    lambda x: style_score(x, friendly_keywords)
)

# ============================================================
# 3. RESPONSE LENGTH (CHAR + TOKEN)
# ============================================================

df["response_char_length"] = df["model_output"].apply(lambda x: len(str(x)))

# ============================================================
# 4. COMPOSITE QUALITY SCORE (optional but powerful)
# ============================================================

df["quality_score"] = (
    df["reference_similarity"] * 0.4 +
    df["style_pass"].astype(int) * 0.2 +
    (1 - df["hard_fail"].astype(int)) * 0.2 +
    (1 - df["leakage_severity"] / 3) * 0.2
)

# ============================================================
# 5. SUMMARY TABLE UPDATE
# ============================================================

soft_summary = (
    df.groupby(["model_type", "model_name", "prompt_type", "persona"])
    .agg(
        avg_leakage_severity=("leakage_severity", "mean"),
        avg_pressure_style_score=("pressure_style_score", "mean"),
        avg_friendly_style_score=("friendly_style_score", "mean"),
        avg_char_length=("response_char_length", "mean"),
        avg_quality_score=("quality_score", "mean"),
    )
    .reset_index()
)

soft_summary = soft_summary.round(3)

print("\n===== SOFT METRICS SUMMARY =====")
display(soft_summary)

# ============================================================
# SAVE
# ============================================================

SAVE_PATH = "/content/drive/MyDrive/project_05/soft_metrics_summary.csv"

soft_summary.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print("\nsaved to:", SAVE_PATH)

In [ ]:
# ============================================================
# NEW STYLE PASS (Improved Rule-based)
# ============================================================

import re

# ------------------------------------------------------------
# Rule 정의
# ------------------------------------------------------------

PRESSURE_RULES = {
    "max_sentences": 3,
    "max_questions": 1,
    "must_include_keywords_any": [
        ["왜", "어떻게", "어떤", "구체적으로", "근거", "수치", "사례", "경험"]
    ],
    "forbidden_phrases": [
        "높이 평가",
        "훌륭",
        "좋은 경험",
        "인정합니다",
    ],
    "must_end_with_question": True,
    "forbidden_emojis": True,
    "min_korean_ratio": 0.8,
}

FRIENDLY_RULES = {
    "max_sentences": 4,
    "exact_questions": 1,

    # 하나 이상 포함
    "must_include_one_of": {
        "context_expansion": [
            "상황",
            "역할",
            "행동",
            "결과",
            "이유",
            "근거",
            "사례",
            "배운 점",
            "경험",
            "과정",
        ]
    },

    # 너무 빡세지 않도록 완화
    "must_include_softener": [
        "말씀",
        "좋습니다",
        "네",
        "그렇군요",
        "감사합니다",
        "부탁드립니다",
        "주실 수",
        "해주실 수",
    ],

    "forbidden_phrases": [
        "높이 평가",
        "좋은 경험",
        "훌륭",
    ],

    "must_end_with_question": True,
    "min_korean_ratio": 0.8,
}

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE,
)

def sentence_count(text):
    text = str(text)
    s = re.split(r"[.!?]+", text)
    s = [x for x in s if x.strip()]
    return len(s)

def question_count(text):
    text = str(text)
    return text.count("?") + text.count("？")

def ends_with_question(text):
    text = str(text).strip()
    return text.endswith("?") or text.endswith("？")

# ------------------------------------------------------------
# Pressure Rule
# ------------------------------------------------------------

def pressure_rule_pass(text):

    text = str(text)

    if sentence_count(text) > PRESSURE_RULES["max_sentences"]:
        return False

    if question_count(text) > PRESSURE_RULES["max_questions"]:
        return False

    if not any(
        any(k in text for k in group)
        for group in PRESSURE_RULES["must_include_keywords_any"]
    ):
        return False

    if any(
        p in text
        for p in PRESSURE_RULES["forbidden_phrases"]
    ):
        return False

    if PRESSURE_RULES["must_end_with_question"]:
        if not ends_with_question(text):
            return False

    if PRESSURE_RULES["forbidden_emojis"]:
        if emoji_pattern.search(text):
            return False

    if korean_ratio(text) < PRESSURE_RULES["min_korean_ratio"]:
        return False

    return True

# ------------------------------------------------------------
# Friendly Rule
# ------------------------------------------------------------

def friendly_rule_pass(text):

    text = str(text)

    if sentence_count(text) > FRIENDLY_RULES["max_sentences"]:
        return False

    if question_count(text) != FRIENDLY_RULES["exact_questions"]:
        return False

    # context keyword 하나 이상
    if not any(
        k in text
        for k in FRIENDLY_RULES["must_include_one_of"]["context_expansion"]
    ):
        return False

    # softener 하나 이상
    if not any(
        k in text
        for k in FRIENDLY_RULES["must_include_softener"]
    ):
        return False

    if any(
        p in text
        for p in FRIENDLY_RULES["forbidden_phrases"]
    ):
        return False

    if FRIENDLY_RULES["must_end_with_question"]:
        if not ends_with_question(text):
            return False

    if korean_ratio(text) < FRIENDLY_RULES["min_korean_ratio"]:
        return False

    return True

# ============================================================
# new_style_pass 계산
# ============================================================

df["new_style_pass"] = False

pressure_mask = df["persona"] == "pressure_interviewer"
friendly_mask = df["persona"] == "friendly_interviewer"

df.loc[pressure_mask, "new_style_pass"] = (
    df.loc[pressure_mask, "model_output"]
      .apply(pressure_rule_pass)
)

df.loc[friendly_mask, "new_style_pass"] = (
    df.loc[friendly_mask, "model_output"]
      .apply(friendly_rule_pass)
)

# ============================================================
# Summary
# ============================================================

new_style_summary = (
    df.groupby(["model_name", "prompt_type", "persona"])
      .agg(
          new_style_pass_rate=("new_style_pass", "mean")
      )
      .reset_index()
)

new_style_summary["new_style_pass_rate"] *= 100
new_style_summary["new_style_pass_rate"] = (
    new_style_summary["new_style_pass_rate"].round(2)
)

display(new_style_summary)


In [ ]:
QUESTION_ENDINGS = [
    "?", "요?", "까?", "나요?",
    "주세요", "주세요.", "바랍니다", "바랍니다.",
    "주시겠습니까", "주시겠습니까?",
    "말씀해주세요", "설명해주세요",
]

def ends_with_question(text):
    text = text.strip()
    return any(text.endswith(e) for e in QUESTION_ENDINGS)

# ============================================================
# STYLE SCORE + RULE PASS RATE
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Rule Detail
# ------------------------------------------------------------

def pressure_rule_detail(text):

    text = str(text)

    return {
        "sentence":
            sentence_count(text) <= PRESSURE_RULES["max_sentences"],

        "question":
            question_count(text) <= PRESSURE_RULES["max_questions"],

        "keyword":
            any(
                any(k in text for k in group)
                for group in PRESSURE_RULES["must_include_keywords_any"]
            ),

        "forbidden":
            not any(
                p in text
                for p in PRESSURE_RULES["forbidden_phrases"]
            ),

        "end":
            ends_with_question(text),

        "emoji":
            emoji_pattern.search(text) is None,

        "korean":
            korean_ratio(text) >= PRESSURE_RULES["min_korean_ratio"],
    }


def friendly_rule_detail(text):

    text = str(text)

    return {
        "sentence":
            sentence_count(text) <= FRIENDLY_RULES["max_sentences"],

        "question":
            question_count(text) == FRIENDLY_RULES["exact_questions"],

        "context":
            any(
                k in text
                for k in FRIENDLY_RULES["must_include_one_of"]["context_expansion"]
            ),

        "softener":
            any(
                k in text
                for k in FRIENDLY_RULES["must_include_softener"]
            ),

        "forbidden":
            not any(
                p in text
                for p in FRIENDLY_RULES["forbidden_phrases"]
            ),

        "end":
            ends_with_question(text),

        "korean":
            korean_ratio(text) >= FRIENDLY_RULES["min_korean_ratio"],
    }


# ------------------------------------------------------------
# Style Score
# ------------------------------------------------------------

def pressure_style_score(text):
    detail = pressure_rule_detail(text)
    return sum(detail.values()) / len(detail)


def friendly_style_score(text):
    detail = friendly_rule_detail(text)
    return sum(detail.values()) / len(detail)


# ============================================================
# style_score 계산
# ============================================================

df["style_score"] = 0.0

pressure_mask = df["persona"] == "pressure_interviewer"
friendly_mask = df["persona"] == "friendly_interviewer"

df.loc[pressure_mask, "style_score"] = (
    df.loc[pressure_mask, "model_output"]
      .fillna("")
      .apply(pressure_style_score)
)

df.loc[friendly_mask, "style_score"] = (
    df.loc[friendly_mask, "model_output"]
      .fillna("")
      .apply(friendly_style_score)
)


# ============================================================
# Average Style Score Summary
# ============================================================

style_summary = (
    df.groupby(["model_name", "prompt_type", "persona"])
      .agg(
          avg_style_score=("style_score", "mean")
      )
      .reset_index()
)

style_summary["avg_style_score"] *= 100
style_summary["avg_style_score"] = (
    style_summary["avg_style_score"].round(2)
)

print("===== Average Style Score =====")
display(style_summary)


# ============================================================
# Rule Pass Rate
# ============================================================

records = []

for _, row in df.iterrows():

    text = "" if pd.isna(row["model_output"]) else str(row["model_output"])

    if row["persona"] == "pressure_interviewer":
        detail = pressure_rule_detail(text)
    else:
        detail = friendly_rule_detail(text)

    detail.update({
        "model_name": row["model_name"],
        "prompt_type": row["prompt_type"],
        "persona": row["persona"],
    })

    records.append(detail)

rule_df = pd.DataFrame(records)


# ============================================================
# Rule Pass Rate Summary
# ============================================================

rule_pass_summary = (
    rule_df
    .groupby(["model_name", "prompt_type", "persona"])
    .mean(numeric_only=True)
    .mul(100)
    .round(2)
    .reset_index()
)

print("===== Rule Pass Rate (%) =====")
display(rule_pass_summary)

In [ ]:
def korean_ratio(text):
    text = str(text).strip()
    if len(text) == 0:
        return 0

    # 영어/숫자는 ICT 용어로 허용 → valid 포함
    # 중국어는 valid 포함 → 분모 증가 → 비율 낮아짐
    valid = re.findall(r"[가-힣a-zA-Z0-9\u4e00-\u9fff\u3040-\u30ff]", text)
    korean = re.findall(r"[가-힣]", text)

    if len(valid) == 0:
        return 0
    return len(korean) / len(valid)